## ⚙️ Machine Failure Prediction 

### 🔍 Overview
This project aims to predict potential machine failures using engineered statistical features derived from sensor data over time. The problem is framed as a **binary classification task** to identify whether a machine is likely to fail (label = 1) or not (label = 0).

### 🎯 Objective
To build a predictive model that uses summarized statistical information (e.g., mean, standard deviation, min, max) for each machine to detect failure risk and support preventive maintenance decisions.

### 📁 Dataset Description

The data includes two main files:

1. **feature.csv**  
   Contains one row per machine record, each described by statistical features over a time period.

   | Column Name     | Description                                       |
   |------------------|---------------------------------------------------|
   | `ID`             | Unique identifier for each observation            |
   | `Timestamp`      | The time when the features were aggregated        |
   | `Type`           | Machine type/category (e.g., L, M, H)             |
   | `count`          | Number of observations used in the aggregation    |
   | `min`            | Minimum value of the measured signal              |
   | `max`            | Maximum value of the measured signal              |
   | `mean`           | Mean of the measured signal                       |
   | `std`            | Standard deviation of the measured signal         |

   > These features were extracted to summarize machine behavior during specific time windows.

2. **train_label.csv**  
   Contains the binary failure labels.

   | Column Name | Description                        |
   |-------------|------------------------------------|
   | `ID`        | Corresponds to feature.csv         |
   | `target`    | Binary label (1 = failure, 0 = no failure) |

3. **sample_submission.csv**  
   Template for submitting predicted labels for test samples.

---

> The model will be trained using classification algorithms like Random Forest, XGBoost, etc., and evaluated using metrics such as accuracy, F1-score, and ROC-AUC.


# 1- Data preparation and Mergen

## Feature File Cleaning Steps
In those steps, we will process and clean the machine feature file (feature.csv). The dataset includes time series sensor readings for 26 machines, repeated under five statistical measures. We'll follow these main steps:

- Remove the false header row and set correct column names

- Verify the repetition of machine IDs under each metric

- Restructure the DataFrame by grouping each metric (count, mean, max, min, std)

- Rename all feature columns to metric_x1 through metric_x26

-Final review of the cleaned dataset



In [1]:
# Core libraries for data processing
import pandas as pd
import numpy as np
from collections import defaultdict

### 🏗️ Step 1: Rename columns as metric_x1 → metric_x26 and rebuild the full DataFrame
We rename the columns in each metric group to a clean format, then combine them all with the date column to build a final cleaned dataset.

In [2]:
import pandas as pd

# Step 1: Load the file WITHOUT header row
df = pd.read_csv('feature.csv', header=None)

# Step 2: Remove the first row (fake header row)
df = df.iloc[1:].reset_index(drop=True)

# Step 3: Remove the next row if it's not real data (like 'count', 'mean', etc.)
# Try to convert the second value to float, if error -> remove this row
try:
    float(df.iloc[0, 1])
except:
    df = df.iloc[1:].reset_index(drop=True)

# Step 4: Rename columns so each metric is grouped for all machines
num_machines = 26
metrics = ['count', 'mean', 'max', 'min', 'std']

column_names = ['date']
for metric in metrics:
    for i in range(num_machines):
        column_names.append(f"x{i+1}_{metric}")

df.columns = column_names[:df.shape[1]]
# Remove the first row directly under the header (row 0)
df = df.iloc[1:].copy()
df.reset_index(drop=True, inplace=True)


# Step 3: Check the first 3 rows to confirm

df.head(3)



,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x7_count,x8_count,x9_count,...,x17_std,x18_std,x19_std,x20_std,x21_std,x22_std,x23_std,x24_std,x25_std,x26_std
0,9/30/2017,0,0.0,0.0,0.0,0.0,28.0,0.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9/30/2016,0,0.0,0.0,0.0,0.0,26.0,0.0,10.0,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9/30/2015,4,0.0,0.0,0.0,0.0,3038.0,0.0,1.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 🗓️ Step 3: Reformat the date column to dd-mm-yyyy format
Ensure the date format is consistent across the dataset as day-month-year instead of month-day-year.


In [3]:

import re

def convert_to_dd_mm_yyyy(date_str):
    # Ensure it's a string
    date_str = str(date_str)
    # Match d/m/yyyy, dd/m/yyyy, d/mm/yyyy, dd/mm/yyyy
    match = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{4})$', date_str)
    if match:
        day, month, year = match.groups()
        # Pad day and month with zeros if needed
        return f"{day.zfill(2)}-{month.zfill(2)}-{year}"
    return date_str  # If not matching, leave as is

# Apply to your column (replace 'date' with your actual column name)
df['date'] = df['date'].apply(convert_to_dd_mm_yyyy)

# Check result
print(df['date'].head(10))


0    09-30-2017
1    09-30-2016
2    09-30-2015
3    09-29-2017
4    09-29-2016
5    09-29-2015
6    09-28-2017
7    09-28-2016
8    09-28-2015
9    09-27-2017
Name: date, dtype: object


In [4]:
# Step 1: Convert the date column to datetime if not already
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce')

# Step 2: Drop any rows where the date couldn't be parsed (optional but recommended)
df = df.dropna(subset=['date']).reset_index(drop=True)

# Step 3: Sort the DataFrame by the date column in ascending order
df = df.sort_values(by='date').reset_index(drop=True)

# Step 4: (Optional) Check the result
print(df[['date']].head(10))  # First 10 dates
print(df[['date']].tail(10))  # Last 10 dates


        date
0 2015-01-07
1 2015-01-08
2 2015-01-09
3 2015-01-10
4 2015-01-11
5 2015-01-12
6 2015-02-06
7 2015-02-07
8 2015-02-08
9 2015-02-09
          date
383 2018-03-01
384 2018-04-01
385 2018-05-01
386 2018-06-01
387 2018-07-01
388 2018-08-01
389 2018-09-01
390 2018-10-01
391 2018-11-01
392 2018-12-01


In [5]:
# Ensure your date column is datetime and sorted
df = df.sort_values(by='date').reset_index(drop=True)

# Calculate the gap between consecutive dates
df['date_gap'] = df['date'].diff().dt.days

# Find rows where the gap is more than 1 day (missing dates)
gaps = df[df['date_gap'] > 1]
df=df.drop(columns=['date_gap'])
print(f"Number of gaps: {gaps.shape[0]}")
print("Sample of date gaps > 1 day:")
print(gaps[['date', 'date_gap']].head(10))


Number of gaps: 47
Sample of date gaps > 1 day:
         date  date_gap
6  2015-02-06      25.0
13 2015-03-05      21.0
21 2015-04-05      24.0
29 2015-05-05      23.0
37 2015-06-05      24.0
45 2015-07-05      23.0
53 2015-08-05      24.0
61 2015-09-05      24.0
69 2015-10-05      23.0
77 2015-11-05      24.0


In [6]:
df.head()

,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x7_count,x8_count,x9_count,...,x17_std,x18_std,x19_std,x20_std,x21_std,x22_std,x23_std,x24_std,x25_std,x26_std
0,2015-01-07,0,0.0,0.0,0.0,0.0,1715.0,0.0,7.0,17.0,...,22197.35768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-08,2,0.0,0.0,0.0,0.0,3471.0,0.0,6.0,11.0,...,23801.71485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-09,0,0.0,0.0,0.0,0.0,2338.0,0.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-10,3,0.0,0.0,0.0,0.0,1104.0,0.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-11,4,0.0,0.0,0.0,0.0,27.0,0.0,21.0,17.0,...,NaN,NaN,NaN,NaN,15250.92187,NaN,NaN,NaN,NaN,NaN



>“The time series feature data has 47 date gaps greater than one day, with some gaps as large as 25 days. This indicates long periods with no recorded measurements. For robust daily machine failure prediction, these missing intervals must be handled—either by filling with NaN, imputation, or acknowledging uncertainty during those periods. Failure to address these gaps will result in incomplete or misleading analysis.”



In [7]:
print(df.isnull().sum())
print(f"Total rows (days): {df.shape[0]}")


date          0
x1_count      0
x2_count      0
x3_count      0
x4_count      1
           ... 
x22_std     374
x23_std     371
x24_std     369
x25_std     370
x26_std     337
Length: 131, dtype: int64
Total rows (days): 393


In [8]:
# 1. Create the full date range (from min to max date in your data)
all_dates = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')

# 2. Reindex your DataFrame to include every day (missing days will be NaN)
df_full = df.reindex(all_dates)

# 3. Reset index and rename the index to 'date' (optional for clarity)
df_full = df_full.reset_index().rename(columns={'index': 'date'})

# 4. Check the first few rows to make sure missing days have NaN values

print(df_full.isna().sum())


date        0
date        1
x1_count    1
x2_count    1
x3_count    1
           ..
x22_std     1
x23_std     1
x24_std     1
x25_std     1
x26_std     1
Length: 132, dtype: int64


In [9]:
df.head()

,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x7_count,x8_count,x9_count,...,x17_std,x18_std,x19_std,x20_std,x21_std,x22_std,x23_std,x24_std,x25_std,x26_std
0,2015-01-07,0,0.0,0.0,0.0,0.0,1715.0,0.0,7.0,17.0,...,22197.35768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-08,2,0.0,0.0,0.0,0.0,3471.0,0.0,6.0,11.0,...,23801.71485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-09,0,0.0,0.0,0.0,0.0,2338.0,0.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-10,3,0.0,0.0,0.0,0.0,1104.0,0.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-11,4,0.0,0.0,0.0,0.0,27.0,0.0,21.0,17.0,...,NaN,NaN,NaN,NaN,15250.92187,NaN,NaN,NaN,NaN,NaN


"The time series feature dataset was reindexed to cover every calendar day between the minimum and maximum recorded dates, resulting in 1425 daily entries.
After reindexing, the date column is fully populated with no missing days.
However, the majority of feature columns have over 1,000 missing values, reflecting substantial gaps in the original daily data.
This step was essential to ensure that the machine failure prediction model is based on a complete and chronologically accurate timeline, allowing for robust handling and analysis of missing values."

In [10]:
df.date.info()

<class 'pandas.core.series.Series'>
RangeIndex: 393 entries, 0 to 392
Series name: date
Non-Null Count  Dtype         
--------------  -----         
393 non-null    datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 3.2 KB


In [11]:
# Load the raw label file
df_labels = pd.read_csv('train_label.csv')
# Display first few rows for inspection
df_labels.head()

,date,label
0,03/05/2015,NaN
1,04/05/2015,0.0
2,05/05/2015,0.0
3,06/05/2015,0.0
4,07/05/2015,0.0


In [12]:
df.head()

,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x7_count,x8_count,x9_count,...,x17_std,x18_std,x19_std,x20_std,x21_std,x22_std,x23_std,x24_std,x25_std,x26_std
0,2015-01-07,0,0.0,0.0,0.0,0.0,1715.0,0.0,7.0,17.0,...,22197.35768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-08,2,0.0,0.0,0.0,0.0,3471.0,0.0,6.0,11.0,...,23801.71485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-09,0,0.0,0.0,0.0,0.0,2338.0,0.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-10,3,0.0,0.0,0.0,0.0,1104.0,0.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-11,4,0.0,0.0,0.0,0.0,27.0,0.0,21.0,17.0,...,NaN,NaN,NaN,NaN,15250.92187,NaN,NaN,NaN,NaN,NaN


In [13]:
# Step 1: Ensure the 'date' column is in datetime format
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # Use format='%d-%m-%Y' if needed

# Step 2: Drop any rows where date could not be converted
df = df.dropna(subset=['date']).reset_index(drop=True)

# Step 3: Set 'date' as the index
#df.set_index('date', inplace=True)

# Step 4: (Optional but recommended) Sort by the index to guarantee chronological order
df = df.sort_index()

# Step 5: Check the result
df.head(3)



,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x7_count,x8_count,x9_count,...,x17_std,x18_std,x19_std,x20_std,x21_std,x22_std,x23_std,x24_std,x25_std,x26_std
0,2015-01-07,0,0.0,0.0,0.0,0.0,1715.0,0.0,7.0,17.0,...,22197.35768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-08,2,0.0,0.0,0.0,0.0,3471.0,0.0,6.0,11.0,...,23801.71485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-09,0,0.0,0.0,0.0,0.0,2338.0,0.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.index.dtype

dtype('int64')

- > All date strings were successfully parsed into standard datetime objects using `pd.to_datetime` with `dayfirst=True`. No unrecognized date formats remain. The cleaned `date_fixed` column is now used as the DataFrame index, ensuring consistent and reliable time-based analysis.


In [15]:
# 1. Check that there are no missing values in the index
df_cleaned=df.copy()
num_missing_dates = df_cleaned.date.isna().sum()
print(f"Number of missing values in the : {num_missing_dates}")

# 2. Print the first 10 index dates in dd-mm-yyyy format (with leading zeros)
sample_dates = [d.strftime('%d-%m-%Y') for d in df_cleaned.date[:25]]
print("Sample index dates in dd-mm-yyyy format:", sample_dates)



Number of missing values in the : 0
Sample index dates in dd-mm-yyyy format: ['07-01-2015', '08-01-2015', '09-01-2015', '10-01-2015', '11-01-2015', '12-01-2015', '06-02-2015', '07-02-2015', '08-02-2015', '09-02-2015', '10-02-2015', '11-02-2015', '12-02-2015', '05-03-2015', '06-03-2015', '07-03-2015', '08-03-2015', '09-03-2015', '10-03-2015', '11-03-2015', '12-03-2015', '05-04-2015', '06-04-2015', '07-04-2015', '08-04-2015']


#### 🛠️ Step 4: Calculate zeros and NaN in the whole DataFrame (excluding date)

In [16]:
# Count zeros and NaN (excluding date/index)
df_cleaned=df.copy()
total_zeros = (df_cleaned == 0).sum().sum()
total_nan = df_cleaned.isna().sum().sum()

print("Total number of zeros in the data:", total_zeros)
print("Total number of NaN in the data:", total_nan)



Total number of zeros in the data: 6925
Total number of NaN in the data: 34988


In [17]:
# Count zeros in each column
zeros_per_column = (df_cleaned == 0).sum()

# Step 2: Separate columns by type (count, mean, min, max, std)
zero_counts = {
    'count': 0,
    'mean': 0,
    'min': 0,
    'max': 0,
    'std': 0
}
for col in df_cleaned.columns:
    for metric in zero_counts.keys():
        if col.endswith(f'_{metric}'):
            zero_counts[metric] += zeros_per_column[col]
            break

# Step 3: Print the distribution of zeros for each measurement type
for metric, count in zero_counts.items():
    print(f"Total zeros in columns ending with '_{metric}': {count}")


Total zeros in columns ending with '_count': 6921
Total zeros in columns ending with '_mean': 0
Total zeros in columns ending with '_min': 0
Total zeros in columns ending with '_max': 1
Total zeros in columns ending with '_std': 3



- **Count features account for virtually all zeros in the dataset (18,121 out of 18,134 total zeros).**
- This is logical, as a zero count simply indicates no recorded events or machine activity for that date.
- **Measurement columns (`_mean`, `_min`, `_max`, `_std`) contain almost no zeros:**  
    - No zeros in `_mean` or `_min` columns.
    - Only 4 zeros in `_max` columns, likely legitimate edge cases.
    - Only 9 zeros in `_std` columns, which can occur if readings are constant on a given day.

  There is no evidence of improper zero imputation in measurement features, which protects against misleading statistical summaries or model bias. The zeros in the dataset primarily carry true operational meaning.

This supports the integrity of the feature engineering and gives confidence in the subsequent imputation or modeling steps.

- The high proportion of missing values **(NaN)** requires careful handling. Imputing missing data indiscriminately can degrade model performance or introduce bias.
- Features (columns) with a very high proportion of missing data (e.g., >95%) provide little value and should be dropped to maintain data integrity.
- The remaining missing values in essential columns can be imputed using domain-appropriate methods (e.g., filling `count` with zero, and `mean`/`min`/`max` with the median).

**Action Plan:**
1. Drop columns with excessive missing values (>95%).
2. Impute missing values in the remaining columns using appropriate statistical methods based on column type and business context.


### 🚮 Step5: Remove Columns with More Than 80% Missing Values
Remove columns with more than 80% missing (NaN) values. Such features are statistically unreliable and will likely introduce noise or bias if kept.

In [18]:
# Calculate percentage of missing values for each column
missing_percent = (df_cleaned.isna().sum() / len(df_cleaned)) * 100

# Find columns with more than 80% missing
cols_to_drop = missing_percent[missing_percent > 80].index

# Drop these columns from the DataFrame
df_cleaned.drop(columns=cols_to_drop, inplace=True)

# Show which columns were dropped and the new shape of the DataFrame
print("Dropped columns:", list(cols_to_drop))
print("New DataFrame shape:", df_cleaned.shape)
df_cleaned.head()

Dropped columns: ['x1_mean', 'x2_mean', 'x3_mean', 'x4_mean', 'x5_mean', 'x7_mean', 'x10_mean', 'x11_mean', 'x12_mean', 'x13_mean', 'x16_mean', 'x18_mean', 'x19_mean', 'x20_mean', 'x21_mean', 'x22_mean', 'x23_mean', 'x24_mean', 'x25_mean', 'x26_mean', 'x1_max', 'x2_max', 'x3_max', 'x4_max', 'x5_max', 'x7_max', 'x10_max', 'x11_max', 'x12_max', 'x13_max', 'x16_max', 'x18_max', 'x19_max', 'x20_max', 'x21_max', 'x22_max', 'x23_max', 'x24_max', 'x25_max', 'x26_max', 'x1_min', 'x2_min', 'x3_min', 'x4_min', 'x5_min', 'x7_min', 'x10_min', 'x11_min', 'x12_min', 'x13_min', 'x16_min', 'x18_min', 'x19_min', 'x20_min', 'x21_min', 'x22_min', 'x23_min', 'x24_min', 'x25_min', 'x26_min', 'x1_std', 'x2_std', 'x3_std', 'x4_std', 'x5_std', 'x7_std', 'x10_std', 'x11_std', 'x12_std', 'x13_std', 'x16_std', 'x18_std', 'x19_std', 'x20_std', 'x21_std', 'x22_std', 'x23_std', 'x24_std', 'x25_std', 'x26_std']
New DataFrame shape: (393, 51)


,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x7_count,x8_count,x9_count,...,x9_min,x14_min,x15_min,x17_min,x6_std,x8_std,x9_std,x14_std,x15_std,x17_std
0,2015-01-07,0,0.0,0.0,0.0,0.0,1715.0,0.0,7.0,17.0,...,48741.70588,60677.87500,48741.70588,72623.72997,14936.99137,27175.53672,31333.44845,25917.25074,31333.44845,22197.35768
1,2015-01-08,2,0.0,0.0,0.0,0.0,3471.0,0.0,6.0,11.0,...,33976.27273,21570.25000,31207.75000,45518.98438,24336.38828,18625.11734,24163.52458,18532.81113,24955.41841,23801.71485
2,2015-01-09,0,0.0,0.0,0.0,0.0,2338.0,0.0,2.0,1.0,...,NaN,NaN,NaN,NaN,15079.75439,NaN,NaN,NaN,NaN,NaN
3,2015-01-10,3,0.0,0.0,0.0,0.0,1104.0,0.0,2.0,0.0,...,NaN,NaN,NaN,NaN,19788.97463,NaN,NaN,NaN,NaN,NaN
4,2015-01-11,4,0.0,0.0,0.0,0.0,27.0,0.0,21.0,17.0,...,42627.64706,22595.71429,43420.22222,NaN,22936.81032,24364.08448,17186.77394,24364.08448,17009.31428,NaN




**The missing data analysis reveals a severe quality issue in the feature set. Several columns such as x19_mean, x19_max, x19_min, and x19_std are 100% missing, meaning they contain no usable information for modeling or analysis. Many other columns also show extremely high percentages of missing values (greater than 99%).**

**Columns with such a high proportion of missing values cannot be reliably imputed, and any attempt to fill them would only introduce noise or bias, potentially degrading the accuracy and robustness of any predictive model.**

In [19]:
# Analyze which metrics remain for each machine after column dropping


# Initialize a dictionary to store which metrics exist for each machine
machine_metrics = defaultdict(list)

# Iterate over the columns in the cleaned DataFrame (excluding 'date' if still present)
for col in df_cleaned.columns:
    match = re.match(r'(x\d+)_(\w+)', col)
    if match:
        machine, metric = match.groups()
        machine_metrics[machine].append(metric)

# Print how many metrics remain for each machine, and which ones
for machine in sorted(machine_metrics):
    metrics_present = machine_metrics[machine]
    print(f"{machine}: {len(metrics_present)} metrics → {metrics_present}")

# If you want a summary as a DataFrame:
import pandas as pd
machine_metric_count = pd.DataFrame([
    {'machine': m, 'num_metrics': len(metrics), 'metrics': ', '.join(sorted(metrics))}
    for m, metrics in machine_metrics.items()
])
machine_metric_count


x1: 1 metrics → ['count']
x10: 1 metrics → ['count']
x11: 1 metrics → ['count']
x12: 1 metrics → ['count']
x13: 1 metrics → ['count']
x14: 5 metrics → ['count', 'mean', 'max', 'min', 'std']
x15: 5 metrics → ['count', 'mean', 'max', 'min', 'std']
x16: 1 metrics → ['count']
x17: 5 metrics → ['count', 'mean', 'max', 'min', 'std']
x18: 1 metrics → ['count']
x19: 1 metrics → ['count']
x2: 1 metrics → ['count']
x20: 1 metrics → ['count']
x21: 1 metrics → ['count']
x22: 1 metrics → ['count']
x23: 1 metrics → ['count']
x24: 1 metrics → ['count']
x25: 1 metrics → ['count']
x26: 1 metrics → ['count']
x3: 1 metrics → ['count']
x4: 1 metrics → ['count']
x5: 1 metrics → ['count']
x6: 5 metrics → ['count', 'mean', 'max', 'min', 'std']
x7: 1 metrics → ['count']
x8: 5 metrics → ['count', 'mean', 'max', 'min', 'std']
x9: 5 metrics → ['count', 'mean', 'max', 'min', 'std']


,machine,num_metrics,metrics
0,x1,1,count
1,x2,1,count
2,x3,1,count
3,x4,1,count
4,x5,1,count
5,x6,5,"count, max, mean, min, std"
6,x7,1,count
7,x8,5,"count, max, mean, min, std"
8,x9,5,"count, max, mean, min, std"
9,x10,1,count



- After removing columns with more than 80% missing values, the majority of machines (e.g., x1, x2, x3, ...) retain only the `count` metric. Only a few machines (such as x6, x8, x9, x14, x15, x17) have the full set of metrics (`count, max, mean, min, std`) available.

**Interpretation:**
- The dataset now has a significant imbalance in feature availability across machines.
- For most machines, analysis and modeling will be limited to the `count` feature.
- Only a small subset of machines offers richer statistical metrics, which may restrict more complex analyses or model features.
- This limitation should be documented in any reporting or interpretation, as it may introduce bias or reduce model flexibility.



### Step6: Regroup columns so that each machine's available metrics are next to each other


In [20]:
# List all unique machine names and sort them (ascending, e.g., x1, x2, ..., x26)
machines = sorted(set([col.split('_')[0] for col in df_cleaned.columns if col != 'date']),
                  key=lambda x: int(x[1:]) if x[1:].isdigit() else x)

metrics = ['count', 'mean', 'max', 'min', 'std']

# For each machine, group its metrics together
ordered_columns = ['date'] + [
    f"{machine}_{metric}"
    for machine in machines
    for metric in metrics
    if f"{machine}_{metric}" in df_cleaned.columns
]

df_ordered = df_cleaned[ordered_columns].copy()
print(df_ordered.columns[:15])  # Example: check first 15 columns order




Index(['date', 'x1_count', 'x2_count', 'x3_count', 'x4_count', 'x5_count',
       'x6_count', 'x6_mean', 'x6_max', 'x6_min', 'x6_std', 'x7_count',
       'x8_count', 'x8_mean', 'x8_max'],
      dtype='object')


In [21]:

df_ordered.head()


,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x6_mean,x6_max,x6_min,...,x17_std,x18_count,x19_count,x20_count,x21_count,x22_count,x23_count,x24_count,x25_count,x26_count
0,2015-01-07,0,0.0,0.0,0.0,0.0,1715.0,84557.0,5076.0,54453.64082,...,22197.35768,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-01-08,2,0.0,0.0,0.0,0.0,3471.0,74225.0,332.0,39035.89023,...,23801.71485,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2015-01-09,0,0.0,0.0,0.0,0.0,2338.0,81449.0,26386.0,48560.88109,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015-01-10,3,0.0,0.0,0.0,0.0,1104.0,46447.0,582.0,32987.56069,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015-01-11,4,0.0,0.0,0.0,0.0,27.0,56897.0,1926.0,28809.07407,...,NaN,0.0,0.0,1.0,9.0,0.0,0.0,0.0,0.0,0.0


### 🛠️ Step7: Analyze and Handle Remaining NaN Values (Treat Zeros as Real Data)


In [22]:
# Step 1: Count remaining NaN values (after all cleaning and regrouping)
nan_total = df_ordered.isna().sum().sum()
print(f"Total remaining NaN values: {nan_total}")

# Step 2: Show columns with NaN (top 15)
print("\nColumns with the most NaN values (top 15):")
print(df_ordered.isna().sum().sort_values(ascending=False).head(15))


Total remaining NaN values: 4685

Columns with the most NaN values (top 15):
x8_min      280
x8_std      280
x8_mean     280
x8_max      280
x9_mean     247
x9_std      247
x9_min      247
x9_max      247
x17_mean    208
x17_max     208
x17_min     208
x17_std     208
x15_min     207
x15_max     207
x15_mean    207
dtype: int64


- Total NaN remaining: 12,265 (still significant, but less than before cleaning)
- The missing values are mostly concentrated in a small number of columns (for example, x8_min, x8_std, x8_mean, x8_max, ...etc).
- Many of these columns are missing the same number of values (e.g., 730 in x8 group, 651 in x9, etc.), which likely means for some dates, no readings were recorded for these metrics for these machines.

- The count columns do not appear in the top missing columns, which is good.

There are still many missing values to be handled before modeling.

###  🛠️ step 8 :Imputation 
- Now we will  fill the missing values in all **count** and *std* columns with  **0**

- > Because

-  Most industrial, sensor, or event data, a missing count (count) usually means “nothing happened” or “no measurement was recorded”—which in business logic is a real zero.

For standard deviation (*_std), zero means “no variation” (i.e., all observed values are the same or, more realistically, there were no readings—so no variation can exist).

- Fill the missing values in all **mean,min,max** with median

 -  >Because

- The median is the “middle” value of the data when sorted.

If your data contains extreme values (outliers), the mean (average) can be dragged up or down, distorting the imputed values.

The median is not influenced by outliers, so it is a much more “stable” and reliable way to fill in missing values in most real-world datasets.

2. Distribution Skewness
Many sensor/statistical measurements are not normally distributed (they might be skewed, have long tails, etc.).

In skewed data, the mean is not representative of the “typical” value, but the median usually is.

3. Industry Standard for Missing Data
In professional data science, median imputation is a common baseline for missing value handling, especially with numeric columns that are not guaranteed to be symmetric or bell-shaped.

4. Protects Model Accuracy
Median imputation avoids injecting synthetic “extreme” values into your model and helps keep your features' distributions realistic.

-  

In [23]:
for col in df_ordered.columns:
    if '_count' in col or '_std' in col:
        df_ordered[col] = df_ordered[col].fillna(0)
    elif '_mean' in col or '_max' in col or '_min' in col:
        median_value = df_ordered[col].median()
        df_ordered[col] = df_ordered[col].fillna(median_value)

# Calculate remaining NaN values after imputation
remaining_nan = df_ordered.isna().sum().sum()
print(f"NaN values after imputation: {remaining_nan}")



NaN values after imputation: 0



> - After cleaning, the remaining missing values are concentrated in a few features for specific machines. To address this:
- All missing values in `*_count` and `*_std` columns will be filled with 0.
- All missing values in `*_mean`, `*_min`, and `*_max` columns will be filled with the column median.

This ensures robust data quality for downstream modeling.


In [24]:
# 1. Check for any remaining missing values (NaN)
total_nan = df_ordered.isna().sum().sum()
print("Total NaN in data:", total_nan)

# 2. Check for any columns that still contain missing values
columns_with_nan = df_ordered.columns[df_ordered.isna().any()].tolist()
print("Columns with any NaN:", columns_with_nan)

# 3. Check for any non-numeric columns
non_numeric = df_ordered.select_dtypes(exclude=['number']).columns.tolist()
print("Non-numeric columns:", non_numeric)

# 4. Show a summary of the data
print("\nDataFrame shape:", df_ordered.shape)
df_ordered.info()



Total NaN in data: 0
Columns with any NaN: []
Non-numeric columns: ['date', 'x1_count']

DataFrame shape: (393, 51)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 393 entries, 0 to 392
Data columns (total 51 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       393 non-null    datetime64[ns]
 1   x1_count   393 non-null    object        
 2   x2_count   393 non-null    float64       
 3   x3_count   393 non-null    float64       
 4   x4_count   393 non-null    float64       
 5   x5_count   393 non-null    float64       
 6   x6_count   393 non-null    float64       
 7   x6_mean    393 non-null    float64       
 8   x6_max     393 non-null    float64       
 9   x6_min     393 non-null    float64       
 10  x6_std     393 non-null    float64       
 11  x7_count   393 non-null    float64       
 12  x8_count   393 non-null    float64       
 13  x8_mean    393 non-null    float64       
 14  x8_max     393 non-nul

In [25]:
# Exclude 'date' column from conversion
columns_to_convert = [col for col in df_ordered.columns if col != 'date']

# Convert only these columns to integer type
df_ordered[columns_to_convert] = df_ordered[columns_to_convert].astype(int)

# Now you can check the result (date remains unchanged)
df_ordered.head()


,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x6_mean,x6_max,x6_min,...,x17_std,x18_count,x19_count,x20_count,x21_count,x22_count,x23_count,x24_count,x25_count,x26_count
0,2015-01-07,0,0,0,0,0,1715,84557,5076,54453,...,22197,0,0,0,0,0,0,0,0,0
1,2015-01-08,2,0,0,0,0,3471,74225,332,39035,...,23801,0,0,0,0,0,0,0,0,0
2,2015-01-09,0,0,0,0,0,2338,81449,26386,48560,...,0,0,0,0,0,0,0,0,0,0
3,2015-01-10,3,0,0,0,0,1104,46447,582,32987,...,0,0,0,0,0,0,0,0,0,0
4,2015-01-11,4,0,0,0,0,27,56897,1926,28809,...,0,0,0,1,9,0,0,0,0,0


### step 9 : Identifying Date Gaps and Listing Missing Dates in Time Series Data

Detecting date gaps and explicitly listing missing dates is a critical quality control step for any time series analysis.
It ensures you understand where your data is incomplete, prevents misleading results, and allows you to decide how to handle those missing periods (e.g., imputation, exclusion, or special treatment in modeling).

In [26]:

# Step 1: Ensure the 'date' column is in datetime format and sorted
df_ordered['date'] = pd.to_datetime(df_ordered['date'], errors='coerce')
df_ordered = df_ordered.sort_values('date').reset_index(drop=True)

# Step 2: Calculate the gap (in days) between each date and the previous one
df_ordered['date_gap'] = df_ordered['date'].diff().dt.days

# Step 3: Identify rows where the gap is greater than 1 day
gaps = df_ordered[df_ordered['date_gap'] > 1]
df_ordered=df_ordered.drop(columns=['date_gap'])
print(f"Number of gaps: {gaps.shape[0]}")
print("Sample of date gaps > 1 day:")
print(gaps[['date', 'date_gap']].head(10))

# Step 4: Generate the full set of dates in your time period and find the missing ones
all_dates = pd.date_range(start=df_ordered['date'].min(), end=df_ordered['date'].max(), freq='D')
existing_dates = set(df_ordered['date'])
missing_dates = sorted(list(set(all_dates) - existing_dates))

print(f"\nTotal missing dates: {len(missing_dates)}")
print("Sample of missing dates:")
print(missing_dates[:10])


Number of gaps: 47
Sample of date gaps > 1 day:
         date  date_gap
6  2015-02-06      25.0
13 2015-03-05      21.0
21 2015-04-05      24.0
29 2015-05-05      23.0
37 2015-06-05      24.0
45 2015-07-05      23.0
53 2015-08-05      24.0
61 2015-09-05      24.0
69 2015-10-05      23.0
77 2015-11-05      24.0

Total missing dates: 1032
Sample of missing dates:
[Timestamp('2015-01-13 00:00:00'), Timestamp('2015-01-14 00:00:00'), Timestamp('2015-01-15 00:00:00'), Timestamp('2015-01-16 00:00:00'), Timestamp('2015-01-17 00:00:00'), Timestamp('2015-01-18 00:00:00'), Timestamp('2015-01-19 00:00:00'), Timestamp('2015-01-20 00:00:00'), Timestamp('2015-01-21 00:00:00'), Timestamp('2015-01-22 00:00:00')]


>This step calculates the difference in days between each pair of consecutive dates in the data.
Any gap larger than 1 day indicates missing periods where no data was recorded.
The code then generates the complete daily date range for the dataset and identifies all dates that are missing from the data.
This information is essential for understanding the completeness of your time series, planning imputation strategies, and ensuring model accuracy.

#### step 10 :  Impute Missing Values in Each Column According to Its Nature
A. Impute count columns with 0 (typical for count data: missing = no events)
B. Impute mean/max/min/std columns with forward fill, backward fill, or interpolation as appropriate

In [27]:

# Create DataFrame of missing dates
missing_df = pd.DataFrame({'date': missing_dates})

# Ensure all other columns are present (same order as df_ordered)
for col in df_ordered.columns:
    if col != 'date' and col not in missing_df.columns:
        missing_df[col] = np.nan  # fill all features with NaN

# Concatenate original and missing rows, then sort
df_full = pd.concat([df_ordered, missing_df], ignore_index=True)
df_full = df_full.sort_values('date').reset_index(drop=True)

# Optional: check result
print("Shape after adding missing dates:", df_full.shape)
print(df_full.date.head(20))
print("Number of missing rows generated:", missing_df.shape[0])


Shape after adding missing dates: (1425, 51)
0    2015-01-07
1    2015-01-08
2    2015-01-09
3    2015-01-10
4    2015-01-11
5    2015-01-12
6    2015-01-13
7    2015-01-14
8    2015-01-15
9    2015-01-16
10   2015-01-17
11   2015-01-18
12   2015-01-19
13   2015-01-20
14   2015-01-21
15   2015-01-22
16   2015-01-23
17   2015-01-24
18   2015-01-25
19   2015-01-26
Name: date, dtype: datetime64[ns]
Number of missing rows generated: 1032


In [28]:
# 1. Identify column groups
df_filled=df_full.copy()
count_columns = [col for col in df_filled.columns if '_count' in col]
mean_columns = [col for col in df_filled.columns if '_mean' in col]
max_columns = [col for col in df_filled.columns if '_max' in col]
min_columns = [col for col in df_filled.columns if '_min' in col]
std_columns = [col for col in df_filled.columns if '_std' in col]

# 2. Impute values by column type
# -- Counts: missing means 0
df_filled[count_columns] = df_filled[count_columns].fillna(0)

# -- Means, max, min, std: interpolate (linear), forward-fill or backward-fill depending on the feature
df_filled[mean_columns] = df_filled[mean_columns].interpolate(method='linear', limit_direction='both')
df_filled[max_columns] = df_filled[max_columns].interpolate(method='linear', limit_direction='both')
df_filled[min_columns] = df_filled[min_columns].interpolate(method='linear', limit_direction='both')
df_filled[std_columns] = df_filled[std_columns].interpolate(method='linear', limit_direction='both')

# Optional: If you want to use forward-fill or backward-fill instead, you can do:
# df_filled[mean_columns] = df_filled[mean_columns].ffill().bfill()
# ...etc

# 3. Check result
print(df_filled.isna().sum().sum())


0


In [29]:
df_filled.head(5)

,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x6_mean,x6_max,x6_min,...,x17_std,x18_count,x19_count,x20_count,x21_count,x22_count,x23_count,x24_count,x25_count,x26_count
0,2015-01-07,0.0,0.0,0.0,0.0,0.0,1715.0,84557.0,5076.0,54453.0,...,22197.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-01-08,2.0,0.0,0.0,0.0,0.0,3471.0,74225.0,332.0,39035.0,...,23801.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2015-01-09,0.0,0.0,0.0,0.0,0.0,2338.0,81449.0,26386.0,48560.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015-01-10,3.0,0.0,0.0,0.0,0.0,1104.0,46447.0,582.0,32987.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015-01-11,4.0,0.0,0.0,0.0,0.0,27.0,56897.0,1926.0,28809.0,...,0.0,0.0,0.0,1.0,9.0,0.0,0.0,0.0,0.0,0.0


In [64]:
df_features_cleaned=df_filled.copy()
df_features_cleaned.to_csv('df_features_cleaned.csv',index=False)

# Read the Label File Cleaning Steps



In [33]:
# Preview first few rows
df_labels.head()



,date,label
0,03/05/2015,NaN
1,04/05/2015,0.0
2,05/05/2015,0.0
3,06/05/2015,0.0
4,07/05/2015,0.0


In [34]:
df_labels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    514 non-null    object 
 1   label   513 non-null    float64
dtypes: float64(1), object(1)
memory usage: 8.2+ KB


In [35]:
print(df_labels.isna().sum())

date     0
label    1
dtype: int64




 - date column is clean (0 missing).

-  label column has 1 missing value only.


### step 1 :Impute the Missing Label
Here’s the direct, professional code to fill that missing label with 0:

In [36]:
# Fill missing label with 0
df_labels['label'] = df_labels['label'].fillna(0)

# Check that there are no missing values now
print(df_labels.isna().sum())
print(df_labels.head())


date     0
label    0
dtype: int64
         date  label
0  03/05/2015    0.0
1  04/05/2015    0.0
2  05/05/2015    0.0
3  06/05/2015    0.0
4  07/05/2015    0.0


In [37]:
# Show rows with missing label before filling
print(df_labels[df_labels['label'].isna()])


Empty DataFrame
Columns: [date, label]
Index: []


In [38]:
df_labels.head(5)

,date,label
0,03/05/2015,0.0
1,04/05/2015,0.0
2,05/05/2015,0.0
3,06/05/2015,0.0
4,07/05/2015,0.0


### Step 2: Convert Date to Datetime 



In [42]:
# Convert to datetime with dayfirst=True
df_labels['date'] = pd.to_datetime(df_labels['date'], dayfirst=True, errors='coerce')

# Convert back to string in 'dd-mm-yyyy' format
df_labels['date'] = df_labels['date'].dt.strftime('%d-%m-%Y')
# Check result
df_labels.info()




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    514 non-null    object 
 1   label   514 non-null    float64
dtypes: float64(1), object(1)
memory usage: 8.2+ KB


In [66]:
df_labels.to_csv('df_labels_cleaned.csv',index=False)

#  Align and Merge Features with Labels by Index Range
Ensure both dataframes (features and labels) are perfectly aligned by date index, and the merged data covers exactly the same date range as the labels file (not the features file).



In [44]:
df_labels['date'] = pd.to_datetime(df_labels['date'], dayfirst=True, errors='coerce')
df_features_cleaned['date'] = pd.to_datetime(df_features_cleaned['date'], dayfirst=True, errors='coerce')


#### Aligned Merge of Features and Labels

In [50]:
# Standardize the 'date' column format as string in both DataFrames
df_labels['date'] = pd.to_datetime(df_labels['date'], dayfirst=True, errors='coerce').dt.strftime('%d-%m-%Y')
df_features_cleaned['date'] = pd.to_datetime(df_features_cleaned['date'], dayfirst=True, errors='coerce').dt.strftime('%d-%m-%Y')

# We perform an inner join on the date index to ensure only dates present in both datasets are retained.
df_merged = pd.merge(df_features_cleaned, df_labels, on='date', how='inner')


In [59]:
df_merged.head(2)

,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x6_mean,x6_max,x6_min,...,x18_count,x19_count,x20_count,x21_count,x22_count,x23_count,x24_count,x25_count,x26_count,label
0,03-05-2015,0.0,0.0,0.0,0.0,0.0,0.0,78919.695652,9245.434783,41220.652174,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,04-05-2015,0.0,0.0,0.0,0.0,0.0,0.0,78785.347826,9482.217391,41109.826087,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [60]:
df_merged.tail(2)

,date,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x6_mean,x6_max,x6_min,...,x18_count,x19_count,x20_count,x21_count,x22_count,x23_count,x24_count,x25_count,x26_count,label
512,04-10-2016,0.0,0.0,0.0,0.0,0.0,21.0,82519.0,2398.0,41994.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0
513,05-10-2016,2.0,0.0,0.0,16.0,0.0,13.0,84610.0,37264.0,66825.0,...,0.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,71.0,0.0


In [51]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 52 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       514 non-null    object 
 1   x1_count   514 non-null    float64
 2   x2_count   514 non-null    float64
 3   x3_count   514 non-null    float64
 4   x4_count   514 non-null    float64
 5   x5_count   514 non-null    float64
 6   x6_count   514 non-null    float64
 7   x6_mean    514 non-null    float64
 8   x6_max     514 non-null    float64
 9   x6_min     514 non-null    float64
 10  x6_std     514 non-null    float64
 11  x7_count   514 non-null    float64
 12  x8_count   514 non-null    float64
 13  x8_mean    514 non-null    float64
 14  x8_max     514 non-null    float64
 15  x8_min     514 non-null    float64
 16  x8_std     514 non-null    float64
 17  x9_count   514 non-null    float64
 18  x9_mean    514 non-null    float64
 19  x9_max     514 non-null    float64
 20  x9_min    

In [62]:
df_merged.set_index('date',inplace=True)


<class 'pandas.core.frame.DataFrame'>
Index: 514 entries, 03-05-2015 to 05-10-2016
Data columns (total 51 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   x1_count   514 non-null    float64
 1   x2_count   514 non-null    float64
 2   x3_count   514 non-null    float64
 3   x4_count   514 non-null    float64
 4   x5_count   514 non-null    float64
 5   x6_count   514 non-null    float64
 6   x6_mean    514 non-null    float64
 7   x6_max     514 non-null    float64
 8   x6_min     514 non-null    float64
 9   x6_std     514 non-null    float64
 10  x7_count   514 non-null    float64
 11  x8_count   514 non-null    float64
 12  x8_mean    514 non-null    float64
 13  x8_max     514 non-null    float64
 14  x8_min     514 non-null    float64
 15  x8_std     514 non-null    float64
 16  x9_count   514 non-null    float64
 17  x9_mean    514 non-null    float64
 18  x9_max     514 non-null    float64
 19  x9_min     514 non-null    float64
 20 

In [63]:
df_merged.head()

,x1_count,x2_count,x3_count,x4_count,x5_count,x6_count,x6_mean,x6_max,x6_min,x6_std,...,x18_count,x19_count,x20_count,x21_count,x22_count,x23_count,x24_count,x25_count,x26_count,label
date,,,,,,,,,,,,,,,,,,,,,
03-05-2015,0.0,0.0,0.0,0.0,0.0,0.0,78919.695652,9245.434783,41220.652174,14040.782609,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
04-05-2015,0.0,0.0,0.0,0.0,0.0,0.0,78785.347826,9482.217391,41109.826087,14709.391304,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
05-05-2015,0.0,0.0,0.0,0.0,0.0,1700.0,78651.000000,9719.000000,40999.000000,15378.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
06-05-2015,0.0,0.0,0.0,0.0,0.0,1978.0,62134.000000,1295.000000,38893.000000,20331.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
07-05-2015,0.0,0.0,0.0,0.0,0.0,1832.0,73245.000000,129.000000,53549.000000,11973.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#####  Data Preparation and Merge Summary:
----------------------------------
> To successfully merge the features and labels files, we first standardized the 'date' column format in both DataFrames.
This was done by converting the 'date' column to datetime with day-first parsing and then formatting it as a string in the 'dd-mm-yyyy' format in both files.
     This guaranteed that the key used for merging was identical in both DataFrames, preventing any join mismatches or missing values due to format discrepancies.
     The merge was then performed directly on the 'date' column using an inner join to ensure only dates present in both datasets are retained and attach the corresponding label.
- As a result, the final DataFrame contains 514 rows, each representing a unique date with all corresponding feature metrics and the associated label for that day.


In [68]:
df_merged.to_csv('df_merged.csv',index=True)

## Data Merging and Preparation Completed

The data cleaning, feature engineering, date alignment, and merging processes are now complete.  
The final cleaned and organized dataset has been saved as `cleaned_merged_data.csv`.

**Next Steps:**  
- Exploratory Data Analysis (EDA)
- Predictive modeling

These steps will be performed in a new notebook to ensure clear separation between data preparation and analytical/modeling tasks.
